# IAM 기반 Identity 격리를 적용한 Amazon Bedrock AgentCore Runtime 및 AgentCore Memory Agent

## 개요

이 튜토리얼에서는 IAM 기반 Identity propagation이 적용된 Amazon Bedrock AgentCore Memory를 사용하여 대화형 Agent에 안전한 메모리 격리를 구현하는 방법을 살펴봅니다. 인증된 사용자 자격 증명을 기준으로 대화 기록을 자동 분할하고 격리하여 multi-tenant 환경에서 데이터 개인정보 보호와 보안을 보장하는 Memory 지원 Agent를 구축합니다.

메모리 격리는 프로덕션 대화형 Agent의 핵심 보안 요구 사항입니다. 적절한 격리가 없으면 사용자가 다른 사용자의 대화 기록에 액세스하여 개인정보 침해와 데이터 유출이 발생할 수 있습니다. 이 튜토리얼에서는 IAM policy를 활용하여 개별 사용자마다 안전하고 격리된 Memory 공간을 만드는 데 중점을 둡니다.

이 구현은 AgentCore Memory가 AWS IAM과 통합되어 인증된 사용자 자격 증명을 기준으로 메모리 경계를 자동 적용하는 방법을 보여 줍니다.

### 튜토리얼 세부 정보

| 정보         | 세부 정보                                                          |
|---------------------|------------------------------------------------------------------|
| 튜토리얼 유형       | 보안 및 Identity 관리                                   |
| Agent 유형          | 단일 대화형 Agent                                      |
| Agentic Framework   | Strands Agents                                                   |
| LLM 모델           | Anthropic Claude Haiku 4.5                                      |
| 주요 기능        | 메모리 격리, IAM, 사용자 컨텍스트         |
| 사용 SDK            | boto3, bedrock-agentcore, bedrock-agentcore-starter-toolkit      |

### 학습 내용

이 튜토리얼에서는 다음 내용을 학습합니다.
1. AgentCore Memory의 메모리 격리를 위한 IAM policy를 구성하는 방법
2. Agent 호출 과정에서 사용자 Identity를 전달하는 방법
3. 격리된 Memory 공간을 사용하는 Multi-User Agent를 배포하고 테스트하는 방법
4. IAM policy로 서로 다른 사용자 간 메모리 격리를 검증하는 방법


### 아키텍처

이 예제는 IAM 기반 메모리 격리가 적용되어 AgentCore Runtime에 배포된 대화형 Agent를 보여 줍니다.

<div style="text-align:left">
    <img src="architecture.png" width="90%"/>
</div>


## 0. 사전 요구 사항

이 튜토리얼을 실행하려면 다음 항목이 필요합니다.
* Python 3.10 이상
* Bedrock, ECR, IAM 및 Cognito에 적절한 권한이 있는 AWS 자격 증명
* Amazon Bedrock AgentCore SDK 및 종속성

먼저 필요한 라이브러리를 설치합니다.

In [ ]:
!pip install -qUr requirements.txt

### 환경 설정

필요한 라이브러리를 가져오고 환경을 구성합니다. 다음 항목을 사용합니다.
- AWS 서비스 상호작용을 위한 `boto3`
- Agent Memory 관리를 위한 `bedrock_agentcore.memory`
- 인증 설정을 위한 여러 utility 함수

In [ ]:
# 가져오기
import os
import uuid
import logging
from bedrock_agentcore_starter_toolkit.operations.memory.manager import MemoryManager
from utils import (
    setup_cognito_user_pool,
    reauthenticate_users,
    get_user_sub,
    create_agentcore_role,
)

# 구성
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(name)s: %(message)s")
logger = logging.getLogger("runtime-memory-agent")
REGION = os.getenv("AWS_REGION", "us-west-2")

## 1. Amazon Cognito User Pool 생성

이 섹션에서는 Amazon Cognito User Pool과 사용자를 생성합니다. Cognito는 Agent에 사용자 인증 및 Identity 관리를 제공하여 각 사용자의 대화 기록에 해당 사용자만 액세스하도록 보장합니다.

`setup_cognito_user_pool` 함수는 다음을 수행합니다.
1. Cognito User Pool이 없으면 생성
2. 인증용 app client 설정
3. 임시 password를 사용하는 테스트 사용자 2명 생성
4. 테스트용 access token 생성

In [ ]:
print("Setting up Amazon Cognito user pool and users...")
cognito_config = setup_cognito_user_pool(region=REGION)
print("Cognito setup completed ✓")

## 2. Memory 리소스 생성

이 섹션에서는 Agent가 대화 기록을 저장할 Memory 리소스를 생성합니다. Memory를 사용하면 Agent가 과거 상호작용을 기억하고 컨텍스트를 유지하여 시간이 지나도 더 일관된 응답을 제공할 수 있습니다.

이 예제에서는 추가 장기 strategy 없이 간단한 단기 Memory 리소스를 생성합니다. Memory는 모든 대화 메시지를 저장하여 AgentCore Runtime에서 세션이 종료된 후에도 세션을 이어 갈 때 Agent가 이전 상호작용을 기억하도록 합니다.

In [ ]:
# 이 리소스의 고유 식별자 생성
unique_id = str(uuid.uuid4())[:8]
memory_name = f"RuntimeIdentityMemoryAgent_{unique_id}"

# Memory Manager 초기화
memory_manager = MemoryManager(region_name=REGION)

# Memory 생성
print("\n🧠 Creating memory...")
print("   This takes 2-3 minutes...\n")

memory = memory_manager.get_or_create_memory(
    name=memory_name,
    strategies=[],
    description="Memory isolation with IAM example.",
    event_expiry_days=30,  # 선택 사항: 필요에 따라 조정
)

MEMORY_ID = memory.get("id")
print("\n✅ Memory created successfully!")
print(f"   Memory ID: {MEMORY_ID}")
print(f"   Status: {memory.get('status')}")

## 3. Memory 지원 Agent 생성

이 섹션에서는 사용자 지정 Hook으로 Memory가 통합된 Strands Agents framework 기반 Agent를 구축합니다. 이 Agent는 AgentCore Memory에서 메시지를 저장하고 검색하여 대화 컨텍스트를 유지합니다.

> **Memory가 중요한 이유**: AgentCore Runtime의 세션은 일정 시간이 지나면 만료되어 대화 컨텍스트가 삭제됩니다. 대화를 Memory에 저장하면 세션 간에 이전 정보가 유지되므로 오랜 시간이 지난 뒤에도 사용자에게 자연스러운 경험을 제공할 수 있습니다.

### Agent 기능

Agent는 다음 작업을 수행합니다.
1. 각 사용자 및 Assistant 메시지를 Memory에 자동 저장
2. 기존 세션을 이어 갈 때 과거 대화 기록 검색
3. 동일한 사용자와의 여러 상호작용에서 컨텍스트 유지
4. 사용자 Identity 검증을 통해 서로 다른 사용자의 대화 격리

### 구현의 주요 구성 요소

#### 1. Memory Hook Provider
사용자 지정 Hook Provider는 다음을 구현합니다.
- `on_agent_initialized`: Agent 시작 시 trigger되어 AgentCore Memory에서 대화 기록 검색
- `on_message_added`: 대화에 새 메시지가 추가될 때 trigger되어 AgentCore Memory에 저장

#### 2. Agent 초기화
`initialize_agent` 함수는 다음을 수행합니다.
- 올바른 리전으로 Memory Hook 구성
- 적절한 상태 변수(memory_id, actor_id, session_id)로 Agent 설정
- LLM의 system prompt 구성

#### 3. 사용자 검증
`get_user_sub` 함수는 다음을 수행합니다.
- JWKS로 Cognito access token을 검증하고 사용자의 sub(고유 ID)를 반환합니다.

#### 4. Entry Point Handler
runtime_memory_agent 함수는 다음을 수행합니다.
- 입력 payload를 파싱하고 사용자 메시지 추출
- Cognito의 JWT token으로 사용자 Identity 검증
- Agent 초기화 및 세션 추적 관리
- 적절한 컨텍스트로 Agent 호출 처리
- Runtime 환경에 형식이 지정된 응답 반환

Agent 파일을 생성합니다.

In [ ]:
%%writefile runtime_identity_memory_agent.py
import os
import jwt
import ast
import json
import logging
from strands import Agent
from jwt import PyJWKClient
from typing import Dict, Any
from strands.models import BedrockModel
from bedrock_agentcore.memory.session import MemorySessionManager
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands.hooks import AgentInitializedEvent, HookProvider, HookRegistry, MessageAddedEvent
from bedrock_agentcore.memory.constants import StrategyType, ConversationalMessage, MessageRole

# 상세 로깅 구성
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s'
)
logger = logging.getLogger("runtime-memory-agent")

# AgentCore app 초기화
app = BedrockAgentCoreApp()

MODEL_ID = os.getenv('MODEL_ID')
MEMORY_ID = os.getenv('MEMORY_ID')
COGNITO_USER_POOL = os.getenv('COGNITO_USER_POOL')
REGION = os.getenv('AWS_REGION')

# 전역 Agent 인스턴스 - 첫 요청에서 초기화
agent = None

class MemoryHookProvider(HookProvider):

    def __init__(self):
        logger.info(f"Initializing MemoryHookProvider")
        self.memory_session_manager = MemorySessionManager(MEMORY_ID, REGION)
    
    def on_agent_initialized(self, event: AgentInitializedEvent):
        """에이전트가 시작될 때 최근 대화 기록을 불러옵니다."""
        logger.info("Agent initialization hook triggered")
        
        actor_id = event.agent.state.get("actor_id")
        session_id = event.agent.state.get("session_id")
        
        logger.info(f"State values - actor_id: {actor_id}, session_id: {session_id}")
        
        if not all([actor_id, session_id]):
            logger.warning("Missing required state values")
            return
        
        try:
            # 세션 존재 여부 확인
            logger.info(f"Checking if session {session_id} exists...")
            session_exists = False
            try:
                events = self.memory_session_manager.list_events(
                    actor_id=actor_id,
                    session_id=session_id,
                    max_results=1
                )
                session_exists = len(events) > 0
                logger.info(f"Session exists: {session_exists}")
            except Exception as e:
                logger.warning(f"Error checking session existence: {e}")
                session_exists = False
            
            if not session_exists:
                logger.info(f"No existing conversation found for session {session_id}")
                return
            
            # 대화 기록 불러오기
            logger.info(f"Loading conversation history for existing session {session_id}")
            recent_turns = self.memory_session_manager.get_last_k_turns(
                actor_id=actor_id,
                session_id=session_id,
                k=5
            )            

            if recent_turns:
                logger.info(f"✅ Loaded {len(recent_turns)} conversation turns from memory")
                
                # Agent 대화 기록에 메시지 추가
                for turn in reversed(recent_turns):
                    for message in turn:
                        role = message['role'].lower()  # 'user' 또는 'assistant'
                        parsed = ast.literal_eval(message['content']['text'])
                        content = parsed[0]['text']
                        
                        # Agent 메시지 기록에 추가
                        event.agent.messages.append({
                            "role": role,
                            "content": [{"text": content}]
                        })
                        logger.info(f"Added {role} message to history: {content[:50]}...")
                
                logger.info(f"✅ Added {len(event.agent.messages)} messages to conversation history")
            else:
                logger.info("No recent turns found for this session")
                    
        except Exception as e:
            logger.error(f"❌ Memory load error: {e}", exc_info=True)
    
    def on_message_added(self, event: MessageAddedEvent):
        """메시지를 메모리에 저장합니다."""
        logger.info("💬 Message added - storing in memory")
        
        actor_id = event.agent.state.get("actor_id")
        session_id = event.agent.state.get("session_id")
        
        if not all([actor_id, session_id]):
            logger.warning("Missing required state values")
            return
        
        try:
            messages = event.agent.messages
            last_message = messages[-1]
            message_content = str(last_message.get("content", ""))
            if last_message["role"] == "user":
                message_role = MessageRole.USER
            elif last_message["role"] == "assistant":
                message_role = MessageRole.ASSISTANT
            
            self.memory_session_manager.add_turns(
                actor_id=actor_id,
                session_id=session_id,
                messages=[ConversationalMessage(message_content, message_role)]
            )
            logger.info("✅ Message stored")
            
        except Exception as e:
            logger.error(f"❌ Error storing message: {e}")
    
    def register_hooks(self, registry: HookRegistry):
        logger.info("Registering memory hooks")
        registry.add_callback(MessageAddedEvent, self.on_message_added)
        registry.add_callback(AgentInitializedEvent, self.on_agent_initialized)

def initialize_agent(actor_id, session_id):
    """처음 사용할 에이전트를 초기화합니다."""
    global agent
    
    logger.info(f"Initializing agent for actor_id={actor_id}, session_id={session_id}")
    
    # 모델 및 Memory Hook 생성
    logger.info(f"Setting model ID: {MODEL_ID}")
    model = BedrockModel(model_id=MODEL_ID)
    logger.info(f"Creating memory hook")
    memory_hook = MemoryHookProvider()
    
    # 적절한 초기 상태로 Agent 생성
    logger.info("Creating agent with memory hook")
    agent = Agent(
        model=model,
        hooks=[memory_hook],
        system_prompt="You're a helpful, memory-enabled agent deployed on AgentCore Runtime. You can remember previous interactions within the same session. Be friendly and concise in your responses.",
        state={
            "actor_id": actor_id,
            "session_id": session_id
        }
    )
    logger.info(f"✅ Agent initialized with state: {agent.state.get()}")

def get_user_sub(access_token: str, region: str, user_pool_id: str) -> str:
    """
    JWKS로 Cognito 액세스 토큰을 검증하고 사용자의 sub(고유 ID)를 반환합니다.

    :param access_token: JWT 액세스 토큰 문자열
    :param region: Cognito User Pool의 AWS 리전
    :param user_pool_id: Cognito User Pool ID
    :return: 토큰이 유효하면 사용자의 'sub' 클레임
    :raises jwt.InvalidTokenError: 검증에 실패한 경우
    """
    access_token = access_token[7:]
    jwks_url = f"https://cognito-idp.{region}.amazonaws.com/{user_pool_id}/.well-known/jwks.json"
    jwks_client = PyJWKClient(jwks_url)
    signing_key = jwks_client.get_signing_key_from_jwt(access_token)

    decoded = jwt.decode(
        access_token,
        signing_key.key,
        algorithms=["RS256"],
        issuer=f"https://cognito-idp.{region}.amazonaws.com/{user_pool_id}",
        options={"require": ["exp", "iat", "iss", "token_use"]}
    )

    if decoded.get("token_use") != "access":
        raise jwt.InvalidTokenError("Token is not an access token")

    return decoded["sub"]

@app.entrypoint
def runtime_memory_agent(payload, context):
    """
    메모리 지원 에이전트의 기본 진입점입니다.
    
    인자:
        payload: 사용자 데이터가 포함된 입력 페이로드
        context: 세션 정보가 포함된 Runtime 컨텍스트 객체
    """
    global agent
    
    # payload와 context 정보를 모두 기록
    logger.info(f"Received payload: {payload}")
    logger.info(f"Context: {context}")
    logger.info(f"User Sub: {get_user_sub(context.request_headers.get('Authorization'), REGION, COGNITO_USER_POOL)}")
    
    # 필수 값 추출 및 검증
    user_input = payload.get("prompt")
    actor_id = get_user_sub(context.request_headers.get('Authorization'), REGION, COGNITO_USER_POOL)
    session_id = context.session_id  # context에서 session_id 가져오기
    
    # 필수 field 검증
    if user_input is None:
        error_msg = "❌ ERROR: Missing 'prompt' field in payload"
        logger.error(error_msg)
        return error_msg
    
    # 첫 요청에서 Agent 초기화
    if agent is None:
        logger.info("First request - initializing agent")
        initialize_agent(actor_id, session_id)
    else:
        logger.info("Using existing agent instance")
        # Session ID가 변경된 경우 업데이트
        if agent.state.get("session_id") != session_id:
            logger.info(f"Updating session ID to {session_id}")
            agent.state.set("session_id", session_id)
        if agent.state.get("actor_id") != actor_id:
            logger.info(f"Updating actor ID to {actor_id}")
            agent.state.set("actor_id", actor_id)
    
    logger.info(f"Agent System Prompt: {agent.system_prompt}")
    # 사용자 입력으로 Agent 호출
    logger.info(f"Invoking agent with input: {user_input}")
    response = agent(user_input)
    response_text = response.message['content'][0]['text']
    logger.info(f"✅ Agent response: {response_text[:50]}...")
    
    return response_text

if __name__ == "__main__":
    logger.info("Starting AgentCore application")
    app.run()

## 4. AgentCore Runtime에 배포

이 섹션에서는 확장성과 간소화된 운영을 제공하는 관리형 Agent runtime 환경인 Amazon Bedrock AgentCore Runtime에 Agent를 배포합니다. AgentCore Runtime이 복잡한 인프라를 처리하므로 배포가 아니라 Agent logic에 집중할 수 있습니다.

수동 서버 설정과 관리가 필요한 기존 배포 방식과 달리 AgentCore Runtime은 Agent container를 AWS 인프라에 배포하고 호출용 보안 HTTPS endpoint를 제공합니다. 이 접근 방식은 Agent가 수요에 맞춰 확장되고 프로덕션 환경에서 안정적으로 작동하도록 보장합니다.

> 💡 **팁**: AgentCore starter toolkit은 IAM 역할, ECR repository, container build를 포함한 복잡한 배포 단계를 모두 처리합니다.

### 배포 구성

배포 구성을 설정합니다.

In [ ]:
iam_role = create_agentcore_role(agent_name=f"runtime_memory_agent_{unique_id}", region=REGION)

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
import time

agentcore_runtime = Runtime()
agent_name = f"runtime_memory_agent_{unique_id}"

response = agentcore_runtime.configure(
    entrypoint="runtime_identity_memory_agent.py",
    execution_role=iam_role["Role"]["RoleName"],
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=REGION,
    agent_name=agent_name,
    non_interactive=True,
    memory_mode="NO_MEMORY",
    idle_timeout=60,
    request_header_configuration={"requestHeaderAllowlist": ["Authorization"]},
    authorizer_configuration={
        "customJWTAuthorizer": {
            "discoveryUrl": cognito_config.get("discovery_url"),
            "allowedClients": [cognito_config.get("client_id")],
        }
    },
)
response

### Agent 실행

이제 Agent를 AgentCore Runtime에 실행합니다. 이 단계에서는 구성된 Agent를 AgentCore의 관리형 인프라에 배포합니다. 이 과정에서 앞서 생성한 Memory ID, 사용할 모델 ID, AWS 리전, 인증용 Cognito User Pool ID 등 Agent에 필요한 필수 환경 변수도 전달합니다.

배포 후에는 사용자 메시지로 호출할 수 있는 보안 endpoint를 통해 Agent에 액세스할 수 있습니다. Endpoint는 Cognito 인증으로 보호되므로 권한이 있는 사용자만 Agent에 액세스할 수 있습니다.

In [ ]:
launch_result = agentcore_runtime.launch(
    env_vars={
        "MEMORY_ID": MEMORY_ID,
        "MODEL_ID": "us.anthropic.claude-haiku-4-5-20251001-v1:0",
        "AWS_REGION": REGION,
        "COGNITO_USER_POOL": cognito_config["pool_id"],
    }
)

In [ ]:
status_response = agentcore_runtime.status()
status = status_response.endpoint["status"]
end_status = ["READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"]

while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint["status"]
    print(f"Current status: {status}")

if status == "READY":
    print("✅ Agent successfully deployed!")
else:
    print(f"❌ Deployment ended with status: {status}")

account_id = status_response.config.account
role_arn = status_response.config.execution_role
role_name = role_arn.split("/")[-1]
print(f"Account ID is {account_id}")
print(f"Agent role is {role_arn}")
print(f"Role name is {role_name}")

## 5. Agent 테스트

Agent가 배포되었으므로 메시지를 전송하여 이전 상호작용을 기억하는지 테스트합니다. 또한 한 사용자의 대화가 다른 사용자에게 보이지 않도록 서로 다른 사용자의 Memory 컨텍스트가 격리되는지 확인합니다.

**세션 관리에 관한 중요 참고 사항**

- **세션 관리**: Session ID를 제공하지 않으면 AgentCore Runtime이 자동으로 생성하지만, 애플리케이션에서 Session ID를 명시적으로 관리하는 것이 좋습니다. 이를 통해 다음 항목을 더 효과적으로 제어할 수 있습니다.
- 세션 timeout 후 대화 계속
- 적절한 시점에 새 세션 생성(예: 사용자가 새 대화를 시작)
- 동일한 사용자의 여러 병렬 대화 처리
- 애플리케이션 요구 사항에 따른 세션 만료 policy 구현

- **메모리 지속성**: AgentCore Runtime에서 세션이 만료되더라도 동일한 사용자가 새 세션을 시작하면 Agent가 AgentCore Memory에서 이전 대화를 검색할 수 있습니다.


In [ ]:
# 필요한 경우 다시 인증
tokens = reauthenticate_users(cognito_config["client_id"], REGION)
cognito_config["bearer_tokens"]["testuser1"] = tokens["testuser1"]
cognito_config["bearer_tokens"]["testuser2"] = tokens["testuser2"]

In [ ]:
import time
import boto3
import json


def test_user_memory_isolation_with_iam():
    """
    Cognito subject ID를 사용하는 동적 IAM 정책 제어로 사용자 메모리 격리를 테스트합니다.
    """
    print("\n" + "=" * 50)
    print("USER MEMORY ISOLATION WITH IAM POLICY TEST")
    print("=" * 50)

    # IAM client 설정
    iam_client = boto3.client("iam")

    # bearer token 추출
    testuser1_token = cognito_config["bearer_tokens"]["testuser1"]
    testuser2_token = cognito_config["bearer_tokens"]["testuser2"]

    user_pool_id = cognito_config["pool_id"]

    # Cognito sub 가져오기(Actor ID로 사용)
    testuser1_actor_id = get_user_sub(testuser1_token, REGION, user_pool_id)
    testuser2_actor_id = get_user_sub(testuser2_token, REGION, user_pool_id)

    print(f"TestUser1 Cognito Sub (actor ID): {testuser1_actor_id}")
    print(f"TestUser2 Cognito Sub (actor ID): {testuser2_actor_id}")

    # 동적 policy template 정의
    def create_actor_policy(actor_id, policy_name):
        policy_document = {
            "Version": "2012-10-17",
            "Statement": [
                {
                    "Sid": "DynamicActorIdRestriction",
                    "Effect": "Allow",
                    "Action": [
                        "bedrock-agentcore:CreateEvent",
                        "bedrock-agentcore:GetEvent",
                        "bedrock-agentcore:GetMemory",
                        "bedrock-agentcore:GetMemoryRecord",
                        "bedrock-agentcore:ListActors",
                        "bedrock-agentcore:ListEvents",
                        "bedrock-agentcore:ListMemoryRecords",
                        "bedrock-agentcore:ListSessions",
                        "bedrock-agentcore:DeleteEvent",
                        "bedrock-agentcore:DeleteMemoryRecord",
                        "bedrock-agentcore:RetrieveMemoryRecords",
                    ],
                    "Resource": [f"arn:aws:bedrock-agentcore:{REGION}:{account_id}:memory/*"],
                    "Condition": {"StringEquals": {"bedrock-agentcore:actorId": actor_id}},
                }
            ],
        }

        try:
            # Policy 생성 또는 업데이트
            try:
                response = iam_client.create_policy(
                    PolicyName=policy_name,
                    PolicyDocument=json.dumps(policy_document),
                    Description=f"Dynamic actor restriction policy for {actor_id}",
                )
                policy_arn = response["Policy"]["Arn"]
            except iam_client.exceptions.EntityAlreadyExistsException:
                # 기존 policy 업데이트
                policy_arn = f"arn:aws:iam::{account_id}:policy/{policy_name}"
                iam_client.create_policy_version(
                    PolicyArn=policy_arn,
                    PolicyDocument=json.dumps(policy_document),
                    SetAsDefault=True,
                )

            # Policy를 역할에 연결
            iam_client.attach_role_policy(RoleName=role_name, PolicyArn=policy_arn)

            print(f"Policy updated for Cognito sub (actor ID): {actor_id}")
            return policy_arn

        except Exception as e:
            print(f"Error updating policy: {e}")
            raise

    # Policy 초기화 함수 정의
    def detach_policy(policy_arn):
        try:
            iam_client.detach_role_policy(RoleName=role_name, PolicyArn=policy_arn)
            print(f"Policy {policy_arn} detached")
        except Exception as e:
            print(f"Error detaching policy: {e}")

    # 각 테스트 단계의 고유 Session ID 생성
    user1_session_id = f"memory-agent-session-user1-{int(time.time())}"
    user2_session_id = f"memory-agent-session-user2-{int(time.time())}"

    # 이 테스트용 policy 생성
    policy_name = f"actor-restriction-{int(time.time())}"

    try:
        # 1단계: 올바른 policy로 사용자 1의 메모리 지속성 테스트
        print("\n" + "=" * 50)
        print("PHASE 1: USER 1 MEMORY PERSISTENCE TEST")
        print("=" * 50)

        # 사용자 1용 policy 생성
        policy_arn = create_actor_policy(testuser1_actor_id, policy_name)

        # Policy가 propagation될 때까지 대기
        print("Waiting for IAM policy to propagate...")
        time.sleep(10)

        # 1단계: 사용자 1이 초기 정보 공유
        print("\n" + "-" * 50)
        print("STEP 1: User 1 shares information")
        print("-" * 50)

        user1_prompt1 = "My name is John and my favorite color is purple."
        response1 = agentcore_runtime.invoke(
            {"prompt": user1_prompt1},
            session_id=user1_session_id,
            bearer_token=testuser1_token,
        )
        print(f'User 1 prompt: "{user1_prompt1}"')
        print(f'User 1 response: "{response1["response"]}"')

        # 세션이 종료될 때까지 대기(75초)
        print("\nWaiting 75 seconds for session to terminate...")
        time.sleep(75)

        # 2단계: 사용자 1이 정보 회상 요청(policy에 따라 성공해야 함)
        print("\n" + "-" * 50)
        print("STEP 2: User 1 recalls information (should succeed)")
        print("-" * 50)

        user1_prompt2 = "What is my name and favorite color?"
        response2 = agentcore_runtime.invoke(
            {"prompt": user1_prompt2},
            session_id=user1_session_id,
            bearer_token=testuser1_token,
        )
        print(f'User 1 prompt: "{user1_prompt2}"')
        print(f'User 1 response: "{response2["response"]}"')

        # 2단계: Policy 제어를 통한 사용자 2 메모리 격리 테스트
        print("\n" + "=" * 50)
        print("PHASE 2: USER 2 MEMORY ISOLATION TEST")
        print("=" * 50)

        # 3단계: 사용자 2가 정보 공유(Memory가 생성되며 API 호출로 작동해야 함)
        print("\n" + "-" * 50)
        print("STEP 3: User 2 shares information")
        print("-" * 50)

        # 초기 상호작용을 허용하도록 policy를 일시적으로 사용자 2로 변경
        detach_policy(policy_arn)
        policy_arn = create_actor_policy(testuser2_actor_id, policy_name)

        # Policy 업데이트가 propagation될 때까지 대기
        print("Waiting for updated IAM policy to propagate...")
        time.sleep(10)

        user2_prompt1 = "My name is Mary and my favorite food is pasta."
        response3 = agentcore_runtime.invoke(
            {"prompt": user2_prompt1},
            session_id=user2_session_id,
            bearer_token=testuser2_token,
        )
        print(f'User 2 prompt: "{user2_prompt1}"')
        print(f'User 2 response: "{response3["response"]}"')

        # 세션이 종료될 때까지 대기
        print("\nWaiting 75 seconds for session to terminate...")
        time.sleep(75)

        # 4단계: Policy를 사용자 1로 되돌린 뒤 사용자 2가 회상 시도(실패해야 함)
        print("\n" + "-" * 50)
        print("STEP 4: Change policy to User 1, User 2 tries to recall (should fail)")
        print("-" * 50)

        # Policy를 사용자 1로 되돌리기
        detach_policy(policy_arn)
        policy_arn = create_actor_policy(testuser1_actor_id, policy_name)

        # Policy 업데이트가 propagation될 때까지 대기
        print("Waiting for updated IAM policy to propagate…")
        time.sleep(10)

        user2_prompt2 = "What is my name and favorite food?"
        response4 = agentcore_runtime.invoke(
            {"prompt": user2_prompt2},
            session_id=user2_session_id,
            bearer_token=testuser2_token,
        )
        print(f'User 2 prompt: "{user2_prompt2}"')
        print(f'User 2 response: "{response4["response"]}"')
        print("\n Agent should not have access to User 2 Memory as the policy is giving access only to User 1 events")

    finally:
        # 리소스 정리 - policy 제거
        detach_policy(policy_arn)

In [ ]:
test_user_memory_isolation_with_iam()

## 핵심 개념

이 튜토리얼에서는 Bedrock AgentCore로 Memory 지원 Agent를 구축하는 데 필요한 몇 가지 중요 개념을 학습했습니다.

1. **Memory 통합**: AgentCore Memory를 사용하여 세션 전반의 대화 기록을 저장하고, 세션이 만료되어도 Agent가 시간 경과에 따라 컨텍스트를 유지하도록 하는 방법

2. **세션 관리**: Session ID를 사용하여 대화를 구성하고 사용자가 돌아왔을 때 관련 기록을 검색하여 자연스러운 경험을 제공하는 방법

3. **AgentCore 배포**: 확장, 보안, 인프라 관리를 자동으로 처리하는 프로덕션 Runtime 환경에 Agent를 배포하는 방법

4. **Memory Hook**: Memory 서비스와 통합되는 사용자 지정 Hook을 구현하여 Agent 수명 주기의 특정 시점에 대화 기록을 저장하고 검색하는 방법

5. **사용자 Identity 및 개인정보 보호**: 인증을 사용하여 각 사용자의 대화 기록이 비공개로 유지되고 다른 사용자와 격리되도록 하는 방법

이러한 개념은 지속형 Memory와 정교한 대화 관리 기능을 갖춘 더 복잡한 Agent를 구축하는 기반이 됩니다.

## 리소스 정리(선택 사항)

이 튜토리얼에서 생성한 리소스가 더 이상 필요하지 않으면 불필요한 AWS 요금을 방지하기 위해 정리할 수 있습니다.

In [ ]:
# 모든 리소스를 삭제하려는 경우에만 이 셀 실행

# 1. AgentCore Runtime 삭제
if "launch_result" in locals() and hasattr(launch_result, "agent_id"):
    try:
        agentcore_control_client = boto3.client("bedrock-agentcore-control", region_name=REGION)

        runtime_delete_response = agentcore_control_client.delete_agent_runtime(
            agentRuntimeId=launch_result.agent_id,
        )
        print(f"✅ Deleted AgentCore Runtime: {launch_result.agent_id}")
    except Exception as e:
        print(f"❌ Error deleting AgentCore Runtime: {e}")
else:
    print("No AgentCore Runtime to delete")

# 2. ECR repository 삭제
if "launch_result" in locals() and hasattr(launch_result, "ecr_uri"):
    try:
        ecr_client = boto3.client("ecr", region_name=REGION)

        repository_name = launch_result.ecr_uri.split("/")[1]
        response = ecr_client.delete_repository(
            repositoryName=repository_name,
            force=True,  # image가 있어도 강제 삭제
        )
        print(f"✅ Deleted ECR repository: {repository_name}")
    except Exception as e:
        print(f"❌ Error deleting ECR repository: {e}")
else:
    print("No ECR repository to delete")

# 3. Memory resource 삭제
try:
    memory_manager.delete_memory(memory_id=MEMORY_ID)
    print(f"✅ Deleted memory resource: {MEMORY_ID}")
except Exception as e:
    print(f"❌ Error deleting memory resource: {e}")

# 4. Cognito User Pool 및 관련 resource 삭제
if "cognito_config" in locals() and cognito_config and "pool_id" in cognito_config:
    try:
        cognito_client = boto3.client("cognito-idp", region_name=REGION)

        # User Pool ID 가져오기
        pool_id = cognito_config["pool_id"]

        # 모든 User Pool Client 나열 및 삭제
        clients_response = cognito_client.list_user_pool_clients(UserPoolId=pool_id, MaxResults=60)

        for client in clients_response.get("UserPoolClients", []):
            client_id = client["ClientId"]
            cognito_client.delete_user_pool_client(UserPoolId=pool_id, ClientId=client_id)
            print(f"✅ Deleted User Pool Client: {client_id}")

        # User Pool 자체 삭제
        cognito_client.delete_user_pool(UserPoolId=pool_id)
        print(f"✅ Deleted Cognito User Pool: {pool_id}")

    except Exception as e:
        print(f"❌ Error deleting Cognito resources: {e}")
else:
    print("No Cognito resources to delete")


# 5. IAM role 및 모든 version을 삭제하는 함수
def delete_iam_role(role_identifier, region=REGION):
    """
    연결된 모든 정책과 버전을 포함해 IAM 역할을 삭제합니다.

    인자:
        role_identifier (str): IAM 역할의 ARN 또는 이름
        region (str): AWS 리전
    """
    try:
        iam_client = boto3.client("iam", region_name=region)

        # 식별자가 ARN인지 역할 이름인지 확인
        if role_identifier.startswith("arn:aws:iam::"):
            # ARN에서 역할 이름 추출
            role_name = role_identifier.split("/")[-1]
        else:
            role_name = role_identifier

        print(f"Attempting to delete IAM role: {role_name}")

        # 1. 모든 managed policy 분리
        attached_policies = iam_client.list_attached_role_policies(RoleName=role_name)
        for policy in attached_policies.get("AttachedPolicies", []):
            iam_client.detach_role_policy(RoleName=role_name, PolicyArn=policy["PolicyArn"])
            print(f"✅ Detached managed policy: {policy['PolicyArn']}")

        # 2. 모든 inline policy 삭제
        inline_policies = iam_client.list_role_policies(RoleName=role_name)
        for policy_name in inline_policies.get("PolicyNames", []):
            iam_client.delete_role_policy(RoleName=role_name, PolicyName=policy_name)
            print(f"✅ Deleted inline policy: {policy_name}")

        # 3. Role과 연결된 모든 instance profile 삭제
        instance_profiles = iam_client.list_instance_profiles_for_role(RoleName=role_name)
        for profile in instance_profiles.get("InstanceProfiles", []):
            iam_client.remove_role_from_instance_profile(
                InstanceProfileName=profile["InstanceProfileName"], RoleName=role_name
            )
            print(f"✅ Removed role from instance profile: {profile['InstanceProfileName']}")

        # 4. 마지막으로 role 삭제
        iam_client.delete_role(RoleName=role_name)
        print(f"✅ Successfully deleted IAM role: {role_name}")

    except Exception as e:
        print(f"❌ Error deleting IAM role: {e}")


delete_iam_role(role_arn)

print("\n✅ Cleanup complete")